<div dir="rtl" style="text-align:right">
<h1 style="text-align:right"><bdi dir="ltr">Attention</bdi> را خانه‌به‌خانه باز کنیم</h1><p style="text-align:right"><b>پرسش آزمایش:</b> وزن هر منبع از کجا می‌آید و چه چیزی را جابه‌جا می‌کند؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-01/28-qkv.html"><bdi dir="ltr">28-qkv</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/29-scores.html"><bdi dir="ltr">29-scores</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/30-scaling.html"><bdi dir="ltr">30-scaling</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/31-values.html"><bdi dir="ltr">31-values</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html"><bdi dir="ltr">33-mask</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این یک نمونهٔ آموزشی دستی است، نه وزن‌های آموزش‌دیدهٔ پروژه. <bdi dir="ltr">Token</bdi>ها فقط برچسب موقعیت‌اند. همهٔ مرحله‌ها را آشکار می‌سازیم؛ کتابخانهٔ <bdi dir="ltr">Attention</bdi> آماده‌ای فراخوانی نمی‌شود. قبل از اجرا حدس بزنید سطر آخر کدام <bdi dir="ltr">Key</bdi> را سازگارتر می‌بیند.</p>
</div>

In [ ]:
import math
tokens = ["a","b","c"]
embeddings = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])  # (T,C)
W_Q = torch.eye(2)
W_K = torch.tensor([[1.,1.],[2.,0.]])
W_V = torch.tensor([[1.,0.],[0.,2.]])
def attention_math(x):
    q, k, v = x @ W_Q.T, x @ W_K.T, x @ W_V.T
    raw = q @ k.T
    scaled = raw / math.sqrt(q.shape[-1])
    allowed = torch.ones(x.shape[0],x.shape[0],dtype=torch.bool).tril()
    masked = scaled.masked_fill(~allowed, float("-inf"))
    weights = torch.softmax(masked, dim=-1)
    output = weights @ v
    return dict(embeddings=x, Q=q, K=k, V=v, raw_scores=raw,
                scaled_scores=scaled, mask=allowed, masked_scores=masked,
                weights=weights, output=output)
trace = attention_math(embeddings)
print("Tokens:", tokens)
for name, value in trace.items():
    inspect(name, value)
    print(value)
torch.testing.assert_close(trace["weights"].sum(-1), torch.ones(3))
assert torch.count_nonzero(trace["weights"].triu(1)) == 0
manual_last = sum(trace["weights"][-1,j] * trace["V"][j] for j in range(3))
torch.testing.assert_close(manual_last, trace["output"][-1])


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">از جدول عدد به نقشهٔ وزن</h2><p style="text-align:right">هر سطر یک <bdi dir="ltr">Query</bdi> و هر ستون یک <bdi dir="ltr">Key</bdi> است. رنگ، ضریب ترکیب <bdi dir="ltr">Value</bdi> را نشان می‌دهد؛ به‌تنهایی علت کامل پاسخ مدل یا رابطهٔ دستوریِ تضمین‌شده نیست.</p>
</div>

In [ ]:
def heatmap(weights, title, ax):
    plot = ax.imshow(weights.detach().numpy(), vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=range(3), yticks=range(3), xticklabels=tokens, yticklabels=tokens,
           xlabel="Key", ylabel="Query", title=title)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f"{weights[i,j].item():.2f}",ha="center",va="center",
                    color="white" if weights[i,j] > 0.5 else "black")
    return plot

changed = embeddings.clone()
changed[1] = torch.tensor([2.,1.])  # One controlled input change.
new_trace = attention_math(changed)
fig, axes = plt.subplots(1,2,figsize=(8,3))
heatmap(trace["weights"], "Original", axes[0])
heatmap(new_trace["weights"], "Changed token b", axes[1])
plt.tight_layout()
plt.show()
print("Output change:", new_trace["output"]-trace["output"])
torch.testing.assert_close(new_trace["output"][0],trace["output"][0])


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">W_V</code> را دو برابر کنید و تابع را دوباره اجرا کنید. انتظار داریم ضرایب <bdi dir="ltr">Attention</bdi> ثابت بمانند و خروجی دو برابر شود؛ چرا؟ سپس تغییر در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">embeddings</code> را با تغییر در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">W_V</code> مقایسه کنید: در اولی <bdi dir="ltr">Q</bdi> و <bdi dir="ltr">K</bdi> هم می‌توانند عوض شوند. علت ثابت‌ماندن سطر اول در تغییر <bdi dir="ltr">Token</bdi> دوم را در دفتر بعد جدا می‌آزماییم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: اثر تغییر یک <bdi dir="ltr">Value</bdi> را پیش‌بینی کنید</h2>
<p style="text-align:right">با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Q/K</code> ثابت، سهم دقیق یک منبع در همهٔ خروجی‌های <bdi dir="ltr">Attention</bdi> را حساب کنید. پیش‌نیاز: مراحل امتیاز، <bdi dir="ltr">Softmax</bdi> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weights @ V</code> همین دفتر را دیده‌اید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر فقط <bdi dir="ltr">Value</bdi>ِ موقعیت یک تغییر کند، کدام ستون از جدول وزن اندازهٔ اثر آن بر <bdi dir="ltr">Query</bdi>های مختلف را مشخص می‌کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
scores = torch.tensor([[1.,0.,2.],[0.,2.,1.],[2.,1.,0.]])
allowed = torch.ones(3,3,dtype=torch.bool).tril()
weights = scores.masked_fill(~allowed,float('-inf')).softmax(-1)
values = torch.tensor([[1.,0.],[0.,2.],[1.,2.]])
print('fixed weights:',weights)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">predict_value_delta(weights,key,delta)</code> اختلاف خروجی همهٔ <bdi dir="ltr">Query</bdi>ها را وقتی فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">V[key]</code> به اندازهٔ بردار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">delta</code> عوض می‌شود، برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weights</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(Q,S)</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">delta</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(C,)</code> است؛ <bdi dir="ltr">Value</bdi> اصلی برای این حساب لازم نیست.</p>
</div>

In [ ]:
def predict_value_delta(weights, key, delta):
    # TODO
    return None

In [ ]:
def test_exercise():
    delta = torch.tensor([2.,-1.])
    result = predict_value_delta(weights,1,delta)
    if result is None: return False
    changed = values.clone(); changed[1] += delta
    torch.testing.assert_close(result,weights@changed-weights@values)
    torch.testing.assert_close(result[0],torch.zeros(2))
    w = torch.tensor([[0.2,0.3,0.5],[1.,0.,0.]])
    d = torch.tensor([1.,2.,3.,4.])
    torch.testing.assert_close(predict_value_delta(w,2,d),torch.stack((0.5*d,torch.zeros_like(d))))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط شمارهٔ <bdi dir="ltr">Value</bdi> تغییرکرده را میان صفر، یک و دو جابه‌جا کنید؛ بردار تغییر ثابت بماند. کدام خروجی‌ها به‌دلیل <bdi dir="ltr">Mask</bdi> دست‌نخورده می‌مانند؟</p>
</div>

In [ ]:
delta = torch.tensor([2.,-1.])
for key in range(3):
    changed = values.clone(); changed[key] += delta
    print('changed key:',key,'output change:',weights@changed-weights@values)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">کد خراب به‌جای ستون <bdi dir="ltr">Key</bdi>، سطر <bdi dir="ltr">Query</bdi> را انتخاب می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">source_share(weights,key)</code> سهم یک منبع در همهٔ <bdi dir="ltr">Query</bdi>ها را به شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(Q,)</code> برگرداند.</p>
</div>

In [ ]:
print('wrong row for key 1:',weights[1])
print('each output needs the same source column')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def source_share(weights, key):
    # TODO
    return None

In [ ]:
def test_repair():
    result = source_share(weights,1)
    if result is None: return False
    torch.testing.assert_close(result,weights[:,1])
    w = torch.tensor([[1.,2.,3.],[4.,5.,6.]])
    torch.testing.assert_close(source_share(w,2),torch.tensor([3.,6.]))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">این رابطه دربارهٔ ضرب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weights @ v</code> در <bdi dir="ltr">Attention</bdi> پروژه است، وقتی وزن‌ها ثابت‌اند. اگر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">X</code> را تغییر دهید، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Q/K</code> و وزن‌ها نیز ممکن است عوض شوند؛ آن آزمایش دیگر فقط تغییر <bdi dir="ltr">Value</bdi> نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا همین فرمول ساده را نمی‌توان بدون قید به تغییر یک <bdi dir="ltr">Token</bdi> ورودی نسبت داد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-07_attention_math.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>